In [1]:
import os
os.chdir('../')

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

In [5]:
from MLProject.constant import *
from MLProject.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS
        
        create_directories([config.root_dir])
        
        data_validation_config = DataValidationConfig(
            root_dir = config.root_dir,
            STATUS_FILE = config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema = schema
        )
        
        return data_validation_config

In [9]:
import os
from MLProject import logger
import pandas as pd

In [10]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config
        
    def validate_all_columns(self) -> bool:
        try:
            validation_status = None
            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)
            
            all_schema = self.config.all_schema.keys()
            
            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                else:
                    validation_status = True
                with open(self.config.STATUS_FILE, 'w') as f:
                    f.write(f'Validation status: {validation_status}')
            return validation_status
        except Exception as e:
            raise e
                    

In [13]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config = data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise

[2026-03-15 02:32:38,634: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-15 02:32:38,635: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-15 02:32:38,637: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-15 02:32:38,638: INFO: common: Created directory at: artifacts]
[2026-03-15 02:32:38,638: INFO: common: Created directory at: artifacts/data_validation]
